# Chicago TNP Two-Stage Tip Model: ONNX Export and Go Validation
# Chicago TNP 两阶段小费模型：ONNX 导出与 Go 验证

This notebook presents the included deployment package. It verifies the model artifacts without retraining and provides an optional retraining cell for the upstream sampled cache.

本 Notebook 展示随包部署产物。默认无需重新训练即可验证模型，同时提供使用上游抽样缓存重新训练的可选 Cell。


## 1. Environment and paths / 环境与路径

Set `TIP_ONNX_GO_PROJECT_DIR` if Jupyter does not start in this folder.

如果 Jupyter 不是从本目录启动，请设置 `TIP_ONNX_GO_PROJECT_DIR`。


In [ ]:
from pathlib import Path
import json
import os
import subprocess

PROJECT_DIR = Path(
    os.getenv("TIP_ONNX_GO_PROJECT_DIR", Path.cwd())
).expanduser().resolve()

if not (PROJECT_DIR / "model" / "feature_schema.json").exists():
    raise FileNotFoundError(
        "Set TIP_ONNX_GO_PROJECT_DIR to this deployment folder. / "
        "请将 TIP_ONNX_GO_PROJECT_DIR 指向本部署文件夹。"
    )

print("PROJECT_DIR:", PROJECT_DIR)


## 2. Inspect the model contract and metrics / 查看模型约定与指标

The schema fixes the 17-feature order and both ONNX input/output names. The metrics file records the 2024 backtest and Python-to-ONNX parity.

schema 固定 17 个特征顺序和两个 ONNX 模型的输入输出名称。指标文件记录 2024 回测与 Python-ONNX 一致性。


In [ ]:
schema = json.loads(
    (PROJECT_DIR / "model" / "feature_schema.json").read_text(encoding="utf-8")
)
metrics = json.loads(
    (PROJECT_DIR / "model" / "model_metrics.json").read_text(encoding="utf-8")
)

print("Feature count / 特征数量:", schema["feature_count"])
print("Feature order / 特征顺序:", schema["feature_order"])
print(json.dumps(metrics, indent=2))


## 3. Python ONNX validation / Python ONNX 验证

This command loads both ONNX files directly and compares 20 fixed cases with stored references.

该命令直接加载两个 ONNX 文件，并将 20 条固定样本与已保存参考值对比。


In [ ]:
python_bin = PROJECT_DIR / ".venv" / "bin" / "python"
if not python_bin.exists():
    python_bin = Path(os.sys.executable)

command = [
    str(python_bin),
    str(PROJECT_DIR / "python" / "verify_onnx.py"),
    "--model-dir", str(PROJECT_DIR / "model"),
    "--schema", str(PROJECT_DIR / "model" / "feature_schema.json"),
    "--cases", str(PROJECT_DIR / "testdata" / "parity_test_cases.json"),
]
subprocess.run(command, check=True)


## 4. Go parity validation / Go 跨语言一致性验证

A final `PASS` confirms that Go and Python ONNX Runtime match within the configured tolerance.

最后显示 `PASS`，说明 Go 与 Python ONNX Runtime 在设定容限内一致。


In [ ]:
subprocess.run(
    [str(PROJECT_DIR / "run_go_test.sh")],
    cwd=PROJECT_DIR,
    check=True,
)


## 5. Optional retraining / 可选重新训练

Retraining requires the sampled cache generated by the upstream tip notebook. Keep `RUN_RETRAINING=False` when only validating the included artifacts.

重新训练需要上游小费 Notebook 生成的抽样缓存。只验证随包模型时，请保持 `RUN_RETRAINING=False`。


In [ ]:
RUN_RETRAINING = False
TIP_CACHE = Path("/path/to/tip_model_sample_5000_per_month.csv.gz")

if RUN_RETRAINING:
    if not TIP_CACHE.exists():
        raise FileNotFoundError(TIP_CACHE)
    subprocess.run(
        [
            str(python_bin),
            str(PROJECT_DIR / "python" / "train_export.py"),
            "--input", str(TIP_CACHE),
            "--output", str(PROJECT_DIR / "model"),
            "--cases-output",
            str(PROJECT_DIR / "testdata" / "parity_test_cases.json"),
        ],
        check=True,
    )
else:
    print("Using included model artifacts. / 使用随包模型产物。")
